# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [ ]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [ ]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [ ]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [ ]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # constat de la question 3 : des âges à -1, qui veulent dire « âge inconnu »
    # 1) je garde l'info dans une colonne 0/1
    df["age_inconnu"] = (df["age"] == -1).astype(int)
    # 2) et je mets l'âge en manquant pour que -1 ne soit plus lu comme un vrai âge
    df["age"] = df["age"].where(df["age"] != -1)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [ ]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [ ]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


### La soumission


In [ ]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [ ]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [ ]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "yoa-666"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


### L'interface, telle qu'on l'appelle


In [ ]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [ ]:
# 1. la part d'absents sur les lignes de contrôle
part_absents_controle = 100 * yval.mean()

# 2. la référence sur les lignes de contrôle = l'exactitude de « toujours 0 »
ma_reference = 100 * (1 - yval.mean())

# 3. le prédicteur : que des 0 (le patient vient)
pred = np.zeros(len(yval), dtype=int)
print("part d'absents (validation) :", round(part_absents_controle, 2), "%")
print("référence (validation)      :", round(ma_reference, 2), "%")
print("score du prédicteur 'toujours 0' :", round(100 * (pred == yval).mean(), 2), "%")

# 4. l'écart avec la référence calculée sur tout train
ecart_a_train = ma_reference - reference
print("écart avec la référence de train :", round(ecart_a_train, 3), "point")
pourquoi_l_ecart_compte = (f"L'écart est de {ecart_a_train:.3f} point : petit, mais si on compare un modèle "
                           "à la référence de train au lieu de celle de la validation, on se trompe de "
                           "plancher. Quand deux modèles sont à quelques dixièmes l'un de l'autre, ça suffit "
                           "pour croire à un gain qui n'existe pas.")

# 5. le collègue à 83 %
ecart_collegue = 83 - ma_reference
ecart_en_barres = ecart_collegue / erreur_type
reponse_au_collegue = (f"83 % c'est {ecart_collegue:.2f} points au-dessus de la référence ({ma_reference:.2f} %), "
                       f"soit {ecart_en_barres:.1f} barres de bruit (une barre = {erreur_type:.2f} point). ")
if ecart_en_barres > 2:
    reponse_au_collegue += "Ça fait plus de deux barres, donc c'est un vrai résultat, pas du hasard."
else:
    reponse_au_collegue += "Ça fait moins de deux barres, donc ce n'est pas encore un résultat."
print(reponse_au_collegue)

controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [ ]:
# 1. balayage SANS filtre : pour chaque colonne, le taux d'absence par groupe, puis max - min
groupes = {}
lignes = []
for c in mon_train.columns:
    if c == CIBLE:
        continue
    g = mon_train.groupby(c)[CIBLE].agg(["mean", "size"])
    groupes[c] = g
    lignes.append({"colonne": c, "ecart": 100 * (g["mean"].max() - g["mean"].min())})
balayage_brut = pd.DataFrame(lignes).set_index("colonne").sort_values("ecart", ascending=False)
display(balayage_brut.head(10))

# 2. la colonne qui gagne sans filtre
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = balayage_brut["ecart"].iloc[0]
groupes_de_500 = int((groupes[colonne_piegee]["size"] >= 500).sum())
print(colonne_piegee, ":", round(ecart_non_filtre, 1), "points,", groupes_de_500, "groupe(s) de 500 lignes")

# 3. balayage FILTRÉ : on garde seulement les groupes d'au moins 500 lignes, puis on compare
lignes = []
for c, g in groupes.items():
    gros = g[g["size"] >= 500]
    if len(gros) >= 2:
        lignes.append({"colonne": c,
                       "ecart": 100 * (gros["mean"].max() - gros["mean"].min()),
                       "plus_petit_groupe": int(gros["size"].min())})
classement = pd.DataFrame(lignes).set_index("colonne").sort_values("ecart", ascending=False)
display(classement.head(10))
plus_petit_groupe_du_classement = classement["plus_petit_groupe"].min()

# 4. la meilleure après filtre
meilleure_colonne = classement.index[0]
ecart_filtre = classement["ecart"].iloc[0]
print("meilleure après filtre :", meilleure_colonne, round(ecart_filtre, 1), "points")

ce_que_le_critere_ne_protege_pas = (f"Après le filtre, c'est {meilleure_colonne} qui est en tête avec "
                                    f"{ecart_filtre:.1f} points. Le critère enlève les petits groupes, mais il "
                                    "ne dit pas si la colonne est connue AVANT le rendez-vous. ")
if meilleure_colonne in A_ECARTER:
    ce_que_le_critere_ne_protege_pas += ("Or elle est dans A_ECARTER : c'est une info qu'on a seulement après, "
                                         "donc une fuite. Il faut vérifier le sens de la colonne en plus du critère.")
else:
    ce_que_le_critere_ne_protege_pas += ("Il faut donc vérifier en plus qu'elle existe bien au moment de la prise "
                                         "du rendez-vous, sinon c'est une fuite.")

# 5. pourquoi la première gagnait
lecture = (f"{colonne_piegee} gagnait avec {ecart_non_filtre:.0f} points parce qu'elle a "
           f"{mon_train[colonne_piegee].nunique()} valeurs pour {len(mon_train)} lignes : presque chaque groupe "
           "a une seule ligne, donc un taux de 0 % ou 100 %. Par rapport au taux global de "
           f"{100 * mon_train[CIBLE].mean():.1f} %, ça ne veut rien dire, aucun de ses groupes n'a 500 lignes.")
print(lecture)

controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [ ]:
# constat 4 : les âges impossibles (il change une décision)
n_age = int((train["age"] == -1).sum())
taux_age = 100 * train.loc[train["age"] == -1, CIBLE].mean()
constat("des âges impossibles à -1",
        f"{n_age} lignes avec age = -1, soit {100 * n_age / len(train):.2f} %",
        f"taux d'absence {taux_age:.1f} % sur ces lignes contre {100 * train[CIBLE].mean():.1f} % en tout",
        "écarter la valeur : la marquer comme manquante (pas les lignes)")

# constat 5 : le trou de solde_credit (constat négatif)
vide = train["solde_credit"].isna()
t_vide = 100 * train.loc[vide, CIBLE].mean()
t_plein = 100 * train.loc[~vide, CIBLE].mean()
constat("le trou de solde_credit ne dit rien sur l'absence" if abs(t_vide - t_plein) < 1
        else "le trou de solde_credit change le taux d'absence",
        f"{100 * vide.mean():.1f} % de vides, écart de taux {t_vide - t_plein:+.2f} point",
        f"taux {t_vide:.1f} % quand c'est vide contre {t_plein:.1f} % quand c'est rempli",
        "garder la colonne telle quelle, rien à corriger")

import re

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [ ]:
# 1. mes prédictions AVANT de mesurer (en barres)
pred_age = 0.5       # peu de lignes à -1, l'arbre s'en sort déjà
pred_doublons = 0    # peu de doublons, ça ne devrait rien changer
pred_taille = 0.5    # la taille n'a pas l'air liée à l'absence

# la table SANS ma feature, pour comparer à score_interne sur le même découpage
X0 = X_sans_features

# 2. l'âge sur les arbres : avec le défaut = score_interne, sans le défaut = -1 mis en manquant
X_age = X0.copy()
X_age["age"] = X_age["age"].where(X_age["age"] != -1)
m = HistGradientBoostingClassifier(random_state=0).fit(X_age.loc[Xtr.index], ytr)
s_avec_age = score_interne
s_sans_age = 100 * m.score(X_age.loc[Xval.index], yval)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type
print(f"arbres    : avec -1 {s_avec_age:.2f} %  |  corrigé {s_sans_age:.2f} %  |  {age_en_barres:+.2f} barre")

# 3. pareil sur le linéaire (il veut des nombres et pas de trous)
def score_lineaire_de(table):
    t = pd.get_dummies(table, columns=CAT).fillna(0)
    lin = LogisticRegression(solver="newton-cholesky").fit(t.loc[Xtr.index], ytr)
    return 100 * lin.score(t.loc[Xval.index], yval)

X_age_lin = X_age.copy()
X_age_lin["age"] = X_age_lin["age"].fillna(X_age_lin["age"].median())   # pas 0 : la médiane
lin_avec_age = score_lineaire_de(X0)
lin_sans_age = score_lineaire_de(X_age_lin)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type
print(f"linéaire  : avec -1 {lin_avec_age:.2f} %  |  corrigé {lin_sans_age:.2f} %  |  {lin_age_en_barres:+.2f} barre")

# les doublons : je les retire de l'entraînement seulement, même validation
garde = ~train.loc[Xtr.index].duplicated()
n_doublons = int(train.duplicated().sum())
m = HistGradientBoostingClassifier(random_state=0).fit(X0.loc[Xtr.index][garde.values], ytr[garde.values])
doublons_en_barres = (100 * m.score(X0.loc[Xval.index], yval) - score_interne) / erreur_type
print(f"doublons  : {n_doublons} lignes en double, effet {doublons_en_barres:+.2f} barre")

# 4. les verdicts
verdict_age = (f"Arbres : {age_en_barres:+.2f} barre, linéaire : {lin_age_en_barres:+.2f} barre. "
               "Pour un arbre, -1 est juste une valeur sous un seuil, il la range dans une branche à part. "
               "Pour le linéaire, -1 entre dans la somme comme un vrai âge (un bébé), ça tire la prédiction. ")
if abs(age_en_barres) < 2 and abs(lin_age_en_barres) < 2:
    verdict_age += ("Ici les deux écarts sont sous 2 barres : le défaut ne compte pour aucun des deux modèles, "
                    "on le corrige pour avoir une table juste, pas pour le score.")
else:
    verdict_age += ("Au moins un écart dépasse 2 barres : le défaut compte pour ce modèle-là, "
                    "il faut le corriger avant de l'utiliser.")
verdict_doublons = (f"{n_doublons} doublons et {doublons_en_barres:+.2f} barre d'effet : le contrôle ne trouve "
                    "rien qui bouge le score. Ce n'est pas inutile, ça dit que ce défaut n'est pas là où se "
                    "jouent les points, et qu'on peut arrêter de chercher de ce côté.")
print(verdict_age)
print(verdict_doublons)

controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [ ]:
# le modèle de la mise en place, sur les lignes de validation (avec ma feature, comme lui)
Xv = X.loc[Xval.index]

ecarts_1000 = []
ecarts_5000 = []
for i in range(20):
    idx = Xv.sample(1000, random_state=i).index
    ecarts_1000.append(100 * modele.score(Xv.loc[idx], yval.loc[idx]))
    idx = Xv.sample(5000, random_state=i).index
    ecarts_5000.append(100 * modele.score(Xv.loc[idx], yval.loc[idx]))

ecart_type_1000 = np.std(ecarts_1000, ddof=1)
ecart_type_5000 = np.std(ecarts_5000, ddof=1)

p = np.mean(ecarts_1000) / 100
formule_1000 = 100 * np.sqrt(p * (1 - p) / 1000)
p = np.mean(ecarts_5000) / 100
formule_5000 = 100 * np.sqrt(p * (1 - p) / 5000)

rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000

display(pd.DataFrame({"observé": [ecart_type_1000, ecart_type_5000],
                      "formule": [formule_1000, formule_5000],
                      "rapport": [rapport_1000, rapport_5000]}, index=["1 000", "5 000"]).round(3))

lecture = (f"Quand n passe de 1 000 à 5 000, la barre baisse : {ecart_type_1000:.2f} -> {ecart_type_5000:.2f} point, "
           f"la formule annonçait une division par racine de 5 (environ 2,2). Les rapports valent "
           f"{rapport_1000:.2f} et {rapport_5000:.2f}, donc à peu près 1 : la formule marche. Mais 20 tirages "
           "c'est peu pour un écart-type, il est lui-même connu à environ 15 % près, donc un rapport de 0,85 "
           "ou 1,15 ne veut pas dire que la formule est fausse.")
print(lecture)

controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [ ]:
N = len(Xv)
n = 25000
if n >= N:
    n = N - 1   # on ne peut pas tirer plus de lignes qu'il n'y en a
print("lignes de validation :", N, "| tirage de", n)

ecarts_25000 = []
for i in range(20):
    idx = Xv.sample(n, random_state=i).index
    ecarts_25000.append(100 * modele.score(Xv.loc[idx], yval.loc[idx]))

ecart_type_25000 = np.std(ecarts_25000, ddof=1)
p = np.mean(ecarts_25000) / 100
formule_25000 = 100 * np.sqrt(p * (1 - p) / n)
rapport_25000 = ecart_type_25000 / formule_25000

# bonus : correction de population finie
correction = np.sqrt((N - n) / (N - 1))
rapport_corrige = ecart_type_25000 / (formule_25000 * correction)
print(f"observé {ecart_type_25000:.3f} | formule {formule_25000:.3f} | rapport {rapport_25000:.2f}")
print(f"avec la correction ({correction:.2f}) le rapport devient {rapport_corrige:.2f}")

pourquoi_la_formule_deborde = (f"La formule suppose des tirages indépendants dans une population très grande, mais "
                               f"là on tire {n} lignes sur {N} sans remise : deux tirages ont presque les mêmes "
                               f"lignes, donc les scores bougent beaucoup moins. Les rapports passent de "
                               f"{rapport_1000:.2f} (1 000) et {rapport_5000:.2f} (5 000) à {rapport_25000:.2f} "
                               "(25 000) : entre 5 000 et 25 000, on est passé d'une petite partie du jeu à presque "
                               f"tout le jeu. Avec la correction de population finie on retombe à {rapport_corrige:.2f}.")
print(pourquoi_la_formule_deborde)

controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [ ]:
# ma prédiction AVANT de mesurer
pred_pas = ("0,05 : à peu près le même score, mais beaucoup d'arbres (jusqu'à 100). 0,5 : même score, "
            "peu d'arbres car il s'arrête tôt. 2 : moins bon, le pas est trop grand, il corrige trop "
            "et s'arrête vite. Je pense que les scores de 0,05 et 0,5 seront à moins de 2 barres.")

reglages = []
for lr in (0.05, 0.5, 2):
    m = HistGradientBoostingClassifier(learning_rate=lr, random_state=0).fit(X0.loc[Xtr.index], ytr)
    s = 100 * m.score(X0.loc[Xval.index], yval)
    reglages.append({"learning_rate": lr, "score": s, "arbres": int(m.n_iter_),
                     "en_barres": (s - reference) / erreur_type})
display(pd.DataFrame(reglages).round(2))

scores_pas = [r["score"] for r in reglages]
ecart_pas = (max(scores_pas) - min(scores_pas)) / erreur_type
score_separe = (f"Entre le meilleur et le moins bon pas il y a {ecart_pas:.1f} barres. "
                + ("Donc au moins deux scores sont séparés de plus de deux barres."
                   if ecart_pas > 2 else "Donc les trois scores sont dans le bruit, aucun n'est vraiment différent."))
ce_qui_borne = (f"Les arbres construits vont de {min(r['arbres'] for r in reglages)} à "
                f"{max(r['arbres'] for r in reglages)} : l'arrêt anticipé coupe tout seul quand ça ne progresse "
                "plus. Changer le pas change surtout le nombre d'arbres, pas beaucoup le score : ce qui borne "
                "le modèle c'est ce que les données contiennent, pas le pas. Ça confirme en gros ma prédiction.")
print(score_separe)
print(ce_qui_borne)

controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# chaque essai est une fonction (lignes d'entraînement, lignes de validation) -> score entre 0 et 1,
# comme ça je peux le refaire sur un autre découpage à la fin
inutiles = [c for c in ("zone_clinique", "indice_meteo_jour", "score_territorial", "age_mois", "cout_acte")
            if c in X0.columns]
X_moins = X0.drop(columns=inutiles)

def f_base(tr, va):
    return HistGradientBoostingClassifier(random_state=0).fit(X0.loc[tr], y.loc[tr]).score(X0.loc[va], y.loc[va])

def f_donnee(tr, va):
    return HistGradientBoostingClassifier(random_state=0).fit(X_moins.loc[tr], y.loc[tr]).score(X_moins.loc[va], y.loc[va])

def f_reglage(tr, va):
    m = HistGradientBoostingClassifier(max_leaf_nodes=63, random_state=0)
    return m.fit(X0.loc[tr], y.loc[tr]).score(X0.loc[va], y.loc[va])

def f_famille(tr, va):
    m = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0)
    return m.fit(X_num.loc[tr], y.loc[tr]).score(X_num.loc[va], y.loc[va])

prevoir("retirer les colonnes inutiles", "pareil, moins d'une barre",
        "ces colonnes sont constantes, du bruit ou en double : le modèle ne s'en servait pas")
prevoir("arbres plus gros (63 feuilles)", "pareil ou un peu mieux",
        "le modèle peut faire des coupes plus fines, mais on est déjà sur un plateau")
prevoir("forêt aléatoire", "pareil ou un peu moins bien",
        "une autre famille d'arbres, le plafond est dans les données")

essai("retirer les colonnes inutiles", lambda: f_donnee(Xtr.index, Xval.index), "5 colonnes retirées")
essai("arbres plus gros (63 feuilles)", lambda: f_reglage(Xtr.index, Xval.index), "max_leaf_nodes=63")
essai("forêt aléatoire", lambda: f_famille(Xtr.index, Xval.index), "RandomForestClassifier")

lignes = []
for nom, nature, f in (("retirer les colonnes inutiles", "donnée", f_donnee),
                       ("arbres plus gros (63 feuilles)", "réglage", f_reglage),
                       ("forêt aléatoire", "famille", f_famille)):
    e = [e for e in ESSAIS if e["hypothèse"] == nom][0]
    gain = (e["mesuré"] - score_interne) / erreur_type
    lignes.append({"essai": nom, "nature": nature, "attendu": e["attendu"], "mesuré": e["mesuré"],
                   "gain_en_barres": round(gain, 2), "verdict": "réel" if abs(gain) >= 2 else "dans le bruit"})
mes_essais = pd.DataFrame(lignes)
display(mes_essais)

# le meilleur de 3, choisi sur cette validation
meilleur = mes_essais.sort_values("gain_en_barres", ascending=False).iloc[0]
optimisme = ("J'ai pris le meilleur des trois parce qu'il gagne sur CETTE validation : une partie de son gain "
             "peut être de la chance, donc son vrai gain est sûrement plus petit que ce que je mesure. "
             "Je m'attends à ce qu'il baisse sur un autre découpage.")

# le même essai sur un autre découpage (graine 1), comparé au modèle de base sur ce même découpage
fonctions = {"retirer les colonnes inutiles": f_donnee, "arbres plus gros (63 feuilles)": f_reglage,
             "forêt aléatoire": f_famille}
tr1, va1 = train_test_split(X0.index, test_size=0.2, stratify=y, random_state=1)
verifie_par_second_tirage = 100 * (fonctions[meilleur["essai"]](tr1, va1) - f_base(tr1, va1)) / erreur_type
print(f"meilleur essai : {meilleur['essai']} | gain {meilleur['gain_en_barres']:+.2f} barre au 1er découpage, "
      f"{verifie_par_second_tirage:+.2f} au 2e")

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [ ]:
# 1. l'écart entre les deux familles, en barres
rapport = (score_interne - score_lineaire) / erreur_type
print(f"arbres {score_interne:.2f} % | linéaire {score_lineaire:.2f} % | écart {rapport:.1f} barres")
print("une différence" if abs(rapport) >= 2 else "pas une différence (moins de 2 barres)")

# 2. trois coefficients comparables : des indicatrices 0/1 (même unité : « oui ou non »)
coefs = pd.Series(modele_lineaire.coef_[0], index=Xtr_n.columns)
indicatrices = coefs[[c for c in coefs.index if any(c.startswith(k + "_") for k in CAT)]]
trois_coefficients = indicatrices.abs().sort_values(ascending=False).index[:3].tolist()
display(coefs[trois_coefficients].round(3))

# 3. les colonnes en double
pourquoi_pas_les_dupliquees = ("age et age_mois, ou montant_acte et cout_acte, disent la même chose : le modèle "
                               "peut répartir l'effet entre les deux comme il veut (gros + d'un côté, gros - de "
                               "l'autre). Chaque coefficient tout seul ne veut plus rien dire, seule leur somme compte.")

# 4. cause ou pas ?
quartier_provoque = ("Non, un coefficient montre une association dans les données observées, pas une cause : "
                     "un quartier peut juste regrouper des patients plus loin, plus jeunes ou moins assurés, "
                     "et c'est ça qui joue.")

# 5. ce que coûte et rapporte le linéaire
cout_et_rapport = (f"Il coûte de la préparation : indicatrices ({Xtr_n.shape[1]} colonnes au lieu de {X0.shape[1]}), "
                   "trous comblés, et il faudrait remettre à l'échelle. Il rapporte un coefficient signé par colonne, "
                   f"qu'on peut lire. Ici il est à {rapport:.1f} barres des arbres.")

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [ ]:
# ma feature est dans ajouter_features (mise en place) : age_inconnu + l'âge -1 mis en manquant
constat_d_origine = "des âges impossibles à -1"

prevoir("ma feature age_inconnu", "pareil, moins d'une barre",
        "peu de lignes ont age = -1, et l'arbre savait déjà les isoler")
score_avec_feature = remesurer()      # X reçoit la feature une fois, modele est réentraîné
noter("ma feature age_inconnu", "age_inconnu + age -1 -> manquant", "pareil, moins d'une barre", score_avec_feature)
gain_de_ma_feature = (score_avec_feature - score_interne) / erreur_type
print(f"avec ma feature {score_avec_feature:.2f} % | sans {score_interne:.2f} % | gain {gain_de_ma_feature:+.2f} barre")

# le contournement : je prépare la validation à la main et j'oublie de mettre -1 en manquant
a_la_main = X_sans_features.loc[Xval.index].copy()
a_la_main["age_inconnu"] = (a_la_main["age"] == -1).astype(int)
# (oubli : a_la_main["age"] = a_la_main["age"].where(a_la_main["age"] != -1))
score_contourne = 100 * (modele.predict(a_la_main) == yval).mean()
score_bon = 100 * (modele.predict(X.loc[Xval.index]) == yval).mean()
ecart_du_contournement = (score_contourne - score_bon) / erreur_type
print(f"par la chaîne {score_bon:.2f} % | à la main {score_contourne:.2f} % | {ecart_du_contournement:+.2f} barre")
print("lignes prédites différemment :", int((modele.predict(a_la_main) != modele.predict(X.loc[Xval.index])).sum()))
quelle_etape_oubliee = ("J'ai oublié de remplacer age = -1 par une valeur manquante : le modèle a appris avec des "
                        "trous et reçoit des -1. Ça ne plante pas, il n'y a aucun message, c'est ça le bug silencieux.")

# la soumission refaite par predict(), donc par la même chaîne que l'entraînement
soumission = pd.DataFrame({ID: test[ID], CIBLE: predict(test)})
soumission.to_csv("submission.csv", index=False)
print("submission.csv refait :", soumission.shape)

controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [ ]:
# 1. l'écart de remplissage train / test, colonne par colonne
communes = [c for c in train.columns if c in test.columns]
ecarts = (100 * (train[communes].notna().mean() - test[communes].notna().mean())).abs().sort_values(ascending=False)
display(ecarts.head(5).round(1))
colonne_trouvee = ecarts.index[0]
ecart_remplissage = ecarts.iloc[0]
deuxieme = ecarts.index[1]
pourquoi_pas_la_derivee = (f"La deuxième, {deuxieme}, n'a que {ecarts.iloc[1]:.1f} points d'écart contre "
                           f"{ecart_remplissage:.1f} pour {colonne_trouvee} : elle bouge un peu entre train et test "
                           "mais elle est toujours remplie, ce n'est pas une colonne qui disparaît.")

# 2. ce qui reste rempli dans le test
remplissage_du_test = 100 * test[colonne_trouvee].notna().mean()
print(f"{colonne_trouvee} : {100 * train[colonne_trouvee].notna().mean():.1f} % rempli dans train, "
      f"{remplissage_du_test:.1f} % dans test")

# 3. le gain interne, sur MON échantillon
Xm = matrice(mon_train)[0]
ym = mon_train[CIBLE]
Xm_avec = Xm.copy()
if pd.api.types.is_numeric_dtype(mon_train[colonne_trouvee]):
    Xm_avec[colonne_trouvee] = mon_train[colonne_trouvee]
else:
    Xm_avec[colonne_trouvee] = mon_train[colonne_trouvee].fillna("").astype("category")
a, b = train_test_split(Xm.index, test_size=0.2, stratify=ym, random_state=0)

m_sans = HistGradientBoostingClassifier(random_state=0).fit(Xm.loc[a], ym.loc[a])
m_avec = HistGradientBoostingClassifier(random_state=0).fit(Xm_avec.loc[a], ym.loc[a])
s_sans = 100 * m_sans.score(Xm.loc[b], ym.loc[b])
s_avec = 100 * m_avec.score(Xm_avec.loc[b], ym.loc[b])
barre_m = 100 * np.sqrt((s_avec / 100) * (1 - s_avec / 100) / len(b))   # la barre de MA validation
gain_interne = s_avec - s_sans
gain_en_barres = gain_interne / barre_m
print(f"sans {s_sans:.2f} % | avec {s_avec:.2f} % | gain {gain_interne:+.2f} pt = {gain_en_barres:+.1f} barres")

# 4. la simulation : je vide la colonne dans la validation, au taux de trous du test
val_trouee = Xm_avec.loc[b].copy()
trous = val_trouee.sample(frac=1 - remplissage_du_test / 100, random_state=0).index
if str(val_trouee[colonne_trouvee].dtype) == "category":
    if "" not in val_trouee[colonne_trouvee].cat.categories:
        val_trouee[colonne_trouvee] = val_trouee[colonne_trouvee].cat.add_categories([""])
    val_trouee.loc[trous, colonne_trouvee] = ""
else:
    val_trouee.loc[trous, colonne_trouvee] = np.nan
score_simule = 100 * m_avec.score(val_trouee, ym.loc[b])
chute_en_barres = (score_simule - s_avec) / barre_m
print(f"avec la colonne trouée comme dans le test : {score_simule:.2f} % ({chute_en_barres:+.1f} barres)")

# 5. la décision
decision_A_ECARTER = (f"Oui c'était la bonne décision : en interne la colonne rapporte {gain_en_barres:+.1f} barres, "
                      f"mais trouée comme dans le test on tombe à {score_simule:.2f} % contre {s_sans:.2f} % sans elle, "
                      "le gain ne sera jamais rendu. Et la raison n'est pas une fuite : la colonne est connue avant "
                      "le rendez-vous, elle a juste arrêté d'être remplie en production.")
print(decision_A_ECARTER)

controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [ ]:
def suspects(df, cible):
    """Rend une table, une ligne par colonne suspecte : nom, écart de remplissage, signe.
       Signe 1 (le VIDE) : plus de 50 points d'écart de remplissage entre les deux groupes.
       Signe 2 (la VALEUR) : une valeur qui ne vit que dans un seul des deux groupes."""
    absents = df[df[cible] == 1]
    presents = df[df[cible] == 0]
    lignes = []
    for c in df.columns:
        if c in (ID, cible):
            continue
        ecart = abs(100 * absents[c].notna().mean() - 100 * presents[c].notna().mean())
        # signe 1 : rempli d'un côté, vide de l'autre
        if ecart > 50:
            lignes.append({"colonne": c, "ecart_remplissage": round(ecart, 1), "signe": "vide"})
            continue
        # signe 2 : une valeur (vue au moins 30 fois) qui n'existe que dans un groupe
        if df[c].nunique() <= 255:
            compte = pd.crosstab(df[c], df[cible])
            if 0 in compte.columns and 1 in compte.columns:
                seules = compte[((compte[0] == 0) | (compte[1] == 0)) & (compte.sum(axis=1) >= 30)]
                if len(seules) > 0:
                    lignes.append({"colonne": c, "ecart_remplissage": round(ecart, 1),
                                   "signe": "valeur : " + ", ".join(str(v) for v in seules.index[:3])})
    return pd.DataFrame(lignes, columns=["colonne", "ecart_remplissage", "signe"])

table_suspects = suspects(train, CIBLE)
display(table_suspects)

# ce que chaque colonne VEUT DIRE
sens = {
    "motif_absence": ("fuite", "le motif d'absence est écrit après le rendez-vous, une fois qu'on sait que le patient n'est pas venu"),
    "duree_consultation": ("fuite", "une durée de consultation n'existe que si la consultation a eu lieu, donc après le rendez-vous"),
    "constantes_relevees": ("fuite", "les constantes sont prises pendant la consultation, donc seulement si le patient est venu"),
    "canal_prise": ("pas une fuite", "le canal est connu dès la prise du rendez-vous ; son problème c'est qu'il disparaît dans le test"),
}
lignes = []
for c in table_suspects["colonne"]:
    verdict, justification = sens.get(c, ("à discuter",
        f"{c} a l'air connue avant le rendez-vous, mais une de ses valeurs ne vit que d'un côté : "
        "il faut savoir ce que ce code veut dire avant de décider"))
    lignes.append({"colonne": c, "verdict": verdict, "justification": justification})
classement_suspects = pd.DataFrame(lignes)
display(classement_suspects)

# le 3e signe : la sentinelle age = -1
porteuses = train["age"] == -1
sentinelle_trouvee = "age = -1"
lignes_porteuses = int(porteuses.sum())
taux_sur_ces_lignes = 100 * train.loc[porteuses, CIBLE].mean()
taux_global = 100 * train[CIBLE].mean()
print(f"{lignes_porteuses} lignes, taux d'absence {taux_sur_ces_lignes:.1f} % contre {taux_global:.1f} % en tout")
verdict_sentinelle = (f"Défaut de saisie, pas une fuite : -1 est un code pour « âge inconnu », la date de naissance "
                      f"est vide sur les mêmes lignes, ça ne vient pas de l'issue du rendez-vous. Le taux sur ces "
                      f"lignes ({taux_sur_ces_lignes:.1f} % contre {taux_global:.1f} %) est à surveiller, mais sur "
                      f"{lignes_porteuses} lignes seulement.")

controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [ ]:
# 1. mon annonce AVANT de regarder les tranches
annonce = ("Je pense qu'à délai égal le rappel aide un peu (moins d'absences avec rappel), et que l'écart global "
           "vient du fait qu'on envoie le rappel surtout pour les rendez-vous lointains. Je changerais d'avis si, "
           "dans les tranches de délai, les absents restaient plus nombreux avec rappel de plus de 2 barres.")

# la colonne du rappel et le délai
col_rappel = [c for c in mon_train.columns if "rappel" in c.lower() or "sms" in c.lower()][0]
r = mon_train[col_rappel]
if pd.api.types.is_numeric_dtype(r):
    avec = r == 1
else:
    avec = r.astype(str).str.lower().isin(["1", "oui", "true", "yes"])
delai = mon_train["delai_jours"]
print("colonne du rappel :", col_rappel, "|", int(avec.sum()), "avec rappel,", int((~avec).sum()), "sans")

# 2. l'écart global
ecart_global = 100 * (mon_train.loc[avec, CIBLE].mean() - mon_train.loc[~avec, CIBLE].mean())
print(f"écart global (avec - sans) : {ecart_global:+.2f} points")

# 3. par tranches de délai, avec la barre de chaque écart
coupe = pd.cut(delai, [-1, 0, 7, 30, 10000], labels=["le jour même", "1-7 jours", "8-30 jours", "plus de 30 jours"])
tranches = []
tranches_exclues = []
for t in coupe.cat.categories:
    dans = coupe == t
    g_avec = mon_train.loc[dans & avec, CIBLE]
    g_sans = mon_train.loc[dans & ~avec, CIBLE]
    if len(g_avec) < 500 or len(g_sans) < 500:
        tranches_exclues.append({"tranche": t, "avec": len(g_avec), "sans": len(g_sans),
                                 "pourquoi": "un des deux groupes a moins de 500 lignes, on ne compare pas"})
        continue
    p1, p2 = g_avec.mean(), g_sans.mean()
    barre = 100 * np.sqrt(p1 * (1 - p1) / len(g_avec) + p2 * (1 - p2) / len(g_sans))
    ecart = 100 * (p1 - p2)
    tranches.append({"tranche": t, "taux_avec": round(100 * p1, 2), "taux_sans": round(100 * p2, 2),
                     "lignes": len(g_avec) + len(g_sans), "ecart": round(ecart, 2), "barre": round(barre, 2),
                     "en_barres": round(ecart / barre, 1)})
display(pd.DataFrame(tranches))
display(pd.DataFrame(tranches_exclues))

# 4. la variable qui explique
delai_median_avec = delai[avec].median()
delai_median_sans = delai[~avec].median()
variable_qui_explique = (f"le délai : médiane de {delai_median_avec:.0f} jours avec rappel contre "
                         f"{delai_median_sans:.0f} jours sans. Le rappel part surtout pour les rendez-vous lointains, "
                         "qu'on oublie plus, donc l'écart global vient du délai et pas du rappel.")
print(variable_qui_explique)

# 5. l'expérience qui trancherait
experience_qui_trancherait = ("On tire au sort, pour chaque rendez-vous, s'il reçoit un rappel ou non (pile ou face).\n"
                              "On compare ensuite le taux d'absence des deux groupes, avec la barre de bruit.\n"
                              "Comme c'est le hasard qui choisit, les deux groupes ont les mêmes délais, âges, quartiers... "
                              "donc la seule différence c'est le rappel : la comparaison est honnête.")

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [ ]:
# 1. trois hypothèses
hypothese("jour de la semaine du rdv",
          "certains jours (le lundi, le vendredi) on manque plus : signe + pour ces jours",
          "ajouter le jour de la semaine à la table, même modèle, même découpage",
          "contredite si le gain est inférieur à 2 barres de bruit par rapport à score_interne")
hypothese("nombre de rendez-vous passés du patient",
          "un habitué manque moins : signe - quand le nombre monte",
          "compter les rendez-vous précédents par patient_id, score avec / sans",
          "contredite si le taux d'absence ne baisse pas d'au moins 2 barres entre 0 et 3+ rendez-vous passés")
hypothese("plus de 60 ans",
          "les patients âgés viennent plus : signe - sur l'absence",
          "taux d'absence des plus de 60 ans contre les autres, groupes de 500 lignes et plus",
          "contredite si l'écart de taux entre les deux groupes fait moins de 2 barres de bruit")

# 2. j'en teste une, sous le même nom
X_jour = X_sans_features.copy()
X_jour["jour_semaine"] = pd.to_datetime(train["date_rdv"], errors="coerce").dt.dayofweek
prevoir("jour de la semaine du rdv", "un peu mieux",
        "les absences changent un peu selon le jour, mais l'effet doit être petit")
s_jour = essai("jour de la semaine du rdv",
               lambda: HistGradientBoostingClassifier(random_state=0)
                       .fit(X_jour.loc[Xtr.index], ytr).score(X_jour.loc[Xval.index], yval),
               "jour_semaine ajouté")
gain_jour = (s_jour - score_interne) / erreur_type
print(f"gain : {gain_jour:+.2f} barre")
confirmee_ou_contredite = "confirmée" if gain_jour >= 2 else "contredite"
print(confirmee_ou_contredite)

# 3. la suite
if confirmee_ou_contredite == "contredite":
    suite_donnee = (f"Gain de {gain_jour:+.2f} barre, sous les 2 barres annoncées : je ne garde pas le jour de la "
                    "semaine dans le modèle. Ça m'apprend que le calendrier n'est pas là où sont les points, je vais "
                    "plutôt tester l'historique du patient.")
else:
    suite_donnee = (f"Gain de {gain_jour:+.2f} barres, au-dessus des 2 annoncées : je garde la feature, mais je "
                    "la revérifie sur un autre découpage avant de la mettre dans ajouter_features.")

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [ ]:
familles = {
    "id_client": "identifiant",
    "date_defaut": "connue après l'issue",
    "statut_6_mois": "connue après l'issue",
    "montant": "utilisable",
    "canal": "risque de disparition",
    "score_interne": "utilisable",
}
justifications = {
    "id_client": "Il sert juste à retrouver le client, il ne dit rien sur son risque.",
    "date_defaut": "Elle n'est remplie que si le client a fait défaut, donc c'est la réponse elle-même.",
    "statut_6_mois": "Il est relevé six mois après l'octroi, au moment de décider il n'existe pas encore.",
    "montant": "Le montant est connu dès la demande et il décrit bien la demande.",
    "canal": "Il est connu à la demande, mais comme canal_prise chez nous il peut arrêter d'être rempli en production.",
    "score_interne": "Il est calculé au moment de la demande, donc disponible quand on décide.",
}
display(pd.DataFrame({"famille": familles, "justification": justifications}))

COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal',
                   'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition',
                      'utilisable']

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [ ]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
